# Prepare MOSS style training in Colab
Use **Runtime → Change runtime type → GPU**. This notebook installs an isolated uv environment and downloads the selected model assets. It does **not** start training or instantiate pretrained models. The current trainer supports random initialization only; pretrained loading is a separate implementation task.

By default, clone **TheDucker1/sd-scripts**, branch **style_loss**. Commit and push the current MOSS changes there first. All required models are fetched, with ConvRot INT8 Qwen3.5-9B by default and no duplicate BF16 text-encoder download. A successful setup does not establish that full training fits the assigned GPU/RAM.

In [ ]:
import io, json, os, shutil, stat, subprocess, sys, zipfile
from pathlib import Path
if sys.platform != 'linux' or not Path('/content').is_dir():
    raise RuntimeError('This notebook is for Google Colab only; do not run it locally.')
from google.colab import files, userdata
print('Colab guard passed. No installer has run yet.')


In [ ]:
# Default: clone the committed fork. Upload/existing remain optional.
SOURCE = 'git'  # 'git', 'upload', or 'existing'
REPO_URL = 'https://github.com/TheDucker1/sd-scripts'
BRANCH = 'style_loss'
REPO = Path('/content/sd-scripts')
if SOURCE == 'upload':
    uploaded = files.upload()
    archives = [data for name, data in uploaded.items() if name.lower().endswith('.zip')]
    if len(archives) != 1:
        raise RuntimeError('Upload exactly one source-only repository ZIP.')
    destination = Path('/content/moss-upload')
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(archives[0])) as archive:
        if sum(info.file_size for info in archive.infolist()) > 512 * 1024**2:
            raise RuntimeError('Source ZIP exceeds 512 MiB; exclude models, datasets and environments.')
        for info in archive.infolist():
            target = (destination / info.filename).resolve()
            if destination not in target.parents or stat.S_ISLNK(info.external_attr >> 16):
                raise RuntimeError('Unsafe archive entry: ' + info.filename)
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info) as source, target.open('wb') as output:
                    shutil.copyfileobj(source, output)
    candidates = list(destination.rglob('moss_image_train_style.py'))
    if len(candidates) != 1:
        raise RuntimeError('Expected exactly one current MOSS style trainer in the ZIP.')
    REPO = candidates[0].parent
    del archives, uploaded
elif SOURCE == 'git':
    if not REPO_URL:
        raise RuntimeError('Set REPO_URL.')
    if not REPO.exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO)], check=True)
    else:
        print('Reusing the existing source directory; setup will validate it without resetting or pulling it.')
elif SOURCE != 'existing':
    raise ValueError('Unknown SOURCE')
SETUP = REPO / 'tools/colab/setup_moss_style.py'
if not SETUP.is_file():
    raise RuntimeError('Missing setup script; commit and push all new MOSS files to the selected fork/branch first.')
print('Using source:', REPO)


In [ ]:
# One-time authentication for this runtime. Never put your token in code or chat.
# Your HF account must already have access to the gated DINOv3 model.
from getpass import getpass
FORCE_TOKEN_PROMPT = False  # Set True to replace an expired/rejected token.
token = None if FORCE_TOKEN_PROMPT else os.environ.get('HF_TOKEN')
if not token and not FORCE_TOKEN_PROMPT:
    try:
        token = userdata.get('HF_TOKEN')
    except Exception:
        token = None
if not token:
    token = getpass('Hugging Face read token (hidden, this runtime only): ').strip()
if not token or not token.strip():
    raise RuntimeError('An authorized HF token is required for the gated DINOv3 download.')
os.environ['HF_TOKEN'] = token.strip()
print('HF token is available for this runtime. No token value is printed or written to disk.')
del token


In [ ]:
DOWNLOAD_MODELS = True
TEXT_ENCODER = 'int8_convrot'  # Supported profiles: 'int8_convrot' (default), 'bf16'.
# W4A8/GPTQ, GGUF, and arbitrary replacement encoders are not supported here.
PERSIST_MODELS_TO_DRIVE = False
VENV = Path('/content/moss-venv')
MODELS = Path('/content/moss-models')
if PERSIST_MODELS_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    MODELS = Path('/content/drive/MyDrive/moss-models')
print('Environment:', VENV, '\nAssets:', MODELS)


In [ ]:
# This is the only installation/download step. Run it on the Colab VM.
# Setup validates DINOv3 access before downloading any model weights.
command = [sys.executable, str(SETUP), '--repo', str(REPO), '--repo-url', REPO_URL, '--branch', BRANCH, '--venv', str(VENV), '--models-dir', str(MODELS), '--text-encoder', TEXT_ENCODER]
if not DOWNLOAD_MODELS:
    command.append('--skip-models')
subprocess.run(command, cwd=REPO, check=True)


In [ ]:
# Inspect metadata only. These cells do not load checkpoint tensors.
print((VENV / 'moss-runtime.json').read_text())
for name in ['resolved_models.json', 'dataset_stream.json']:
    path = MODELS / name
    if path.exists():
        print(name, '->', path)
print('Use this interpreter for later commands:', VENV / 'bin/python')
print('Preparation only: no training has started.')


The environment is separate from the notebook kernel. Launch later scripts with `/content/moss-venv/bin/python`; shell activation in a previous cell does not persist. `moss-installed-packages.txt` records resolved package versions.

`resolved_models.json` records exact model revisions, local paths, and selected text-encoder loader settings. The default INT8 file is fetched from Comfy-Org/MOSS-Image; full BF16 Qwen weights are excluded. `dataset_stream.json` lists pinned shard URLs, not downloaded images. SigLIP contributes config files and MegaStyle supplies its vision weights. `moss-source.json` in the venv records the code commit.

**Do not point the random-only trainer at these assets yet.** Implementing pretrained loading, the real tokenizer/custom tagger architecture, and memory placement is the next task. Full training capacity must be measured separately. Save checkpoints to persistent storage before the Colab VM expires.